<p align="center">
  <h1 align="center">🍳 Cookbook 04: Visualizations & Reports</h1>
  <p align="center">
    <strong>Generating Beautiful, Publication-Ready Seaborn Charts with GradTracer</strong>
  </p>
</p>

---

This dedicated cookbook focuses on the visual, human-readable outputs provided by GradTracer.

AI agents read JSON, but human data scientists need intuition. GradTracer's visualizer creates standard distributions to clarify model health instantly.

### 1. The 3-Panel Embedding Diagnostic Plot
Visualize Popularity Bias, Gradient SNR, and Weight Velocity distributions.

In [ ]:
import torch, torch.nn as nn
from gradtracer import EmbeddingTracker, plot_embedding_diagnostics

emb = nn.Embedding(1000, 32)
tracker = EmbeddingTracker(emb, name="viz_demo_emb", track_interval=1)
optimizer = torch.optim.Adam(emb.parameters(), lr=0.5)
p = 1.0 / torch.arange(1, 1001).float()
p /= p.sum() # Zipfian distribution

print("Simulating Training...")
for _ in range(500):
    batch = torch.multinomial(p, 64, replacement=True)
    optimizer.zero_grad()
    emb(batch).sum().backward()
    tracker.step()
    optimizer.step()

plot_embedding_diagnostics(tracker, top_k=50)

### 2. GBDT Node Stability (Tree Dynamics Reporting)
For XGBoost/LightGBM, we can report the node-level split gain distributions to see if our trees are actually learning or just echoing noise.

In [ ]:
import xgboost as xgb
import pandas as pd, numpy as np
from gradtracer import TreeDynamicsTracker
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
X, y = pd.DataFrame(data.data, columns=data.feature_names), data.target
dtrain = xgb.DMatrix(X, label=y)

print("Training XGBoost...")
bst = xgb.train({'objective': 'reg:squarederror', 'max_depth': 4}, dtrain, num_boost_round=50, verbose_eval=False)

print("Extracting Tree Dynamics...")
tree_tracker = TreeDynamicsTracker(model=bst)

# ✨ Report will output a summary of tree stability
tree_tracker.report()